In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agentic/long-running-agents-gcp/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 02 · Orchestrator / workers (fan-out, fan-in) and the reflection loop — worked

Two ways to compose LLM steps beyond a single loop: **in parallel** (planner → N workers → aggregator) and **iteratively** (generate → critique → revise).

```mermaid
flowchart LR
  P[planner LLM] -->|N subtasks| T[(Pub/Sub topic /<br/>N Cloud Tasks)]
  T --> W1[worker] & W2[worker] & W3[worker]
  W1 & W2 & W3 -->|transaction:<br/>results[id]=r, completed+=1| F[(Firestore run doc)]
  F -->|completed == expected| A[aggregate task<br/>named → dedup]
  A --> S[synthesis LLM]
```
The fan-in is the hard part: workers are at-least-once, the counter must be transactional, and the aggregate trigger must be idempotent.

In [1]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))          # repo root when run from notebooks/
sys.path[:0] = [os.path.join(ROOT, "src"), os.path.join(ROOT, "notebooks")]

def show_journal(run):
    print(f"run {run.run_id}  status={run.status.value}  version={run.version}  steps={run.usage.steps}  tokens={run.usage.tokens}  cost=${run.usage.cost_usd:.4f}")
    for s in run.journal:
        out = json.dumps(s.output, default=str)[:70] if s.output is not None else (s.error or "")
        print(f"  [{s.index}] {s.kind.value:<6} {s.status.value:<7} {s.name:<18} key={s.idempotency_key or '-':<20} {out}")

In [2]:
from lragents.core import *
from lragents.patterns import FanOutFanIn, ReflectionLoop

calls = []
def worker(task):                              # imagine: a Cloud Run worker calling Gemini + a search API
    calls.append(task["title"])
    return {"summary": f"findings for {task['title']}"}

subtasks = [{"title": f"vendor-{i}", "instructions": "assess pricing and SLA"} for i in range(6)]
planner    = ScriptedLLM([Decision.final(json.dumps(subtasks))])
aggregator = ScriptedLLM([lambda msgs: Decision.final("SYNTHESIS over " + str(len(json.loads(msgs[0]["content"])["results"])) + " results")])
dispatcher = InMemoryDispatcher()
fan = FanOutFanIn(store=InMemoryRunStore(), planner=planner, aggregator=aggregator, dispatcher=dispatcher,
                  worker=worker, idempotency=InMemoryIdempotencyStore())
run = fan.start("Compare six vendors")
print("fanned out:", dispatcher.pending(), "subtask tasks; expected =", fan.store.get(run.run_id).state["fan"]["expected"])

fanned out: 6 subtask tasks; expected = 6


In [3]:
dispatcher.duplicate_next()                   # Pub/Sub redelivers the first subtask
dispatcher.drain(fan.handle)
final = fan.store.get(run.run_id)
print("status:", final.status.value, "| result:", final.result)
print("worker calls:", calls)
print("aggregate deliveries:", sum(1 for e in dispatcher.delivered if e.kind == "aggregate"))
print("fan state:", {k: v for k, v in final.state["fan"].items() if k != "subtasks"})

status: SUCCEEDED | result: SYNTHESIS over 6 results
worker calls: ['vendor-0', 'vendor-1', 'vendor-2', 'vendor-3', 'vendor-4', 'vendor-5']
aggregate deliveries: 1
fan state: {'expected': 6, 'completed': 6, 'results': {'0': {'summary': 'findings for vendor-0'}, '1': {'summary': 'findings for vendor-1'}, '2': {'summary': 'findings for vendor-2'}, '3': {'summary': 'findings for vendor-3'}, '4': {'summary': 'findings for vendor-4'}, '5': {'summary': 'findings for vendor-5'}}}


Six workers ran once each despite a duplicate delivery; aggregation ran once.

## Crash after commit, before the aggregate is enqueued
The last worker commits `completed == expected`, then dies before enqueuing `aggregate`. Retry → the transaction sees the subtask already recorded (no double count) **but still reports `all_done`**, so it enqueues the named aggregate task again — de-duplicated, harmless.

In [4]:
faults = FaultInjector(); calls.clear()
dispatcher = InMemoryDispatcher()
fan = FanOutFanIn(store=InMemoryRunStore(), planner=ScriptedLLM([Decision.final(json.dumps(subtasks[:2]))]),
                  aggregator=ScriptedLLM([Decision.final("SYNTH")]), dispatcher=dispatcher, worker=worker,
                  idempotency=InMemoryIdempotencyStore(), faults=faults)
run = fan.start("two vendors")
dispatcher.deliver_one(fan.handle)
faults.crash_once_at("after_commit")
dispatcher.drain(fan.handle)
print(fan.store.get(run.run_id).status.value, "| workers ran:", calls, "| dead-letter:", dispatcher.dead_letter)

SUCCEEDED | workers ran: ['vendor-0', 'vendor-1'] | dead-letter: []


## Where it breaks at scale — and the two fixes
Every worker updates the *same* Firestore document. Fine for ~50 subtasks; at hundreds, transaction contention and the ~1 write/s per-document guidance bite.

1. **One doc per subtask + count query** (`runs/{id}/subtasks/{sid}`), orchestrator polls with a delayed task.
2. **Let Cloud Workflows do the join**: a `parallel` `for` loop calling Cloud Run workers; Workflows waits for all branches. Read `infra/workflows/fan_out_fan_in.yaml`.

In [5]:
print(open(os.path.join(ROOT, "infra", "workflows", "fan_out_fan_in.yaml")).read())

# Fan-out / fan-in with Cloud Workflows doing the join.
#
# When the branch count is known up front, let Workflows own the counter:
# `parallel` runs branches concurrently, waits for all of them, and the
# per-branch `retry` block replaces hand-written back-off. Each worker call is
# still at-least-once (retries!) so the worker endpoint must be idempotent on
# (run_id, subtask_id).
#
# Deploy:  gcloud workflows deploy fan-out-fan-in --source=fan_out_fan_in.yaml --service-account=$WF_SA
main:
  params: [input]                       # {run_id, goal, subtasks: [...], service_url}
  steps:
    - init:
        assign:
          - results: {}
    - fan_out:
        parallel:
          shared: [results]
          concurrency_limit: 10
          for:
            value: task
            index: i
            in: ${input.subtasks}
            steps:
              - run_worker:
                  try:
                    call: http.post
                    args:
                      url: ${input.s

## Reflection loop (evaluator–optimizer)
Iterative composition. The stop condition is **code** (`threshold`, `max_iters`), every iteration is a checkpoint, and generator/critic are separate LLM configurations.

In [6]:
gen    = ScriptedLLM([Decision.final("v1: an abstract"), Decision.final("v2: a better abstract"), Decision.final("v3")])
critic = ScriptedLLM([Decision.final(json.dumps({"score": 5, "feedback": "state the contribution first"})),
                      Decision.final(json.dumps({"score": 9, "feedback": "good"}))])
store, dispatcher = InMemoryRunStore(), InMemoryDispatcher()
refl = ReflectionLoop(store=store, generator=gen, critic=critic, dispatcher=dispatcher, threshold=8, max_iters=3,
                      price=PriceCard(0.5, 3.0))
run = refl.start("Write an abstract for a paper on durable agents")
while dispatcher.deliver_one(refl.handle):
    r = store.get(run.run_id)
    print(f"iter={r.state['iter']} scores={r.state['scores']} status={r.status.value} checkpoints={store.save_count}")
print(store.get(run.run_id).result)

iter=0 scores=[] status=RUNNING checkpoints=1
iter=1 scores=[5] status=RUNNING checkpoints=2
iter=1 scores=[5] status=RUNNING checkpoints=3
iter=2 scores=[5, 9] status=SUCCEEDED checkpoints=4
{'draft': 'v2: a better abstract', 'score': 9, 'iterations': 2}


## Takeaways
* Fan-out is easy; **fan-in is a distributed counter** → transaction + idempotent trigger.
* Side effects live *outside* the transaction (transactions re-run).
* Prefer a managed join (Cloud Workflows `parallel`) when the branch count is known up front.
* Iterative loops need deterministic exits and per-iteration checkpoints.